In [2]:
import numpy as np
import pandas as pd

from scipy.stats import skew
from sklearn.preprocessing import PowerTransformer
from sklearn.feature_selection import mutual_info_classif

In [ ]:

df= pd.read_csv("../../integrated/multi_turn/data/total/features_before_selection(7).csv")

In [4]:
def transform_feature(series, transform):

    if transform == "original":
        return series
    
    #for the values >0 and i care bout small values to be same and large values compressed 
    elif transform == "log1p":

        transformed = []

        for value in series:

            if value < 0:
                value = 0

            transformed.append(np.log(1 + value))

        return np.array(transformed)

    # here i want to emphasize large values and small values to be same
    elif transform == "square":

        transformed = []

        for value in series:
            transformed.append(value * value)

        return np.array(transformed)

    # here i compress the info to be either 0 or 1
    elif transform == "binarize":

        transformed = []

        for value in series:

            if value > 0:
                transformed.append(1.0)
            else:
                transformed.append(0.0)

        return np.array(transformed)

    #here convert to a more gaussian dirstribution handle negative values and zeros find best transformation 
    #is finding the best lambda automatically via maximum likelihood optimization
    elif transform == "yeo-johnson":

        transformer = PowerTransformer(
            method="yeo-johnson",
            standardize=False
        )

        values = np.asarray(series).reshape(-1, 1)

        transformed = transformer.fit_transform(values)

        return transformed.flatten()

    return series

In [5]:
target_column = "label"

feature_columns = [
    col
    for col in df.columns
    if col not in ["label", "conv_id", "turn_id"]
]

In [ ]:
results = []

transformations = [
    "original",
    "log1p",
    "square",
    "binarize",
    "yeo-johnson"
]

for feature in feature_columns:

    print(f"Testing {feature}")

    best_score = -1
    best_transform = None

    for transform in transformations:

        try:

            transformed = transform_feature(
                df[feature],
                transform
            )

            X = np.array(transformed).reshape(-1, 1)
            #decide depending on mutual information score which transformation is best 
            score = mutual_info_classif(
                X,
                df[target_column],
                random_state=42
            )[0]

            results.append({
                "feature": feature,
                "transform": transform,
                "score": score
            })

            if score > best_score:
                best_score = score
                best_transform = transform

        except Exception as e:

            print(
                f"Failed {feature} with {transform}: {e}"
            )

    print(
        f"Best transformation = {best_transform}"
    )

Testing toxicity_score
Best transformation = yeo-johnson
Testing threat_score
Best transformation = yeo-johnson
Testing topic_drift_score
Best transformation = original
Testing drift_acceleration
Best transformation = yeo-johnson
Testing origin_drift
Best transformation = yeo-johnson
Testing drift_momentum
Best transformation = yeo-johnson
Testing persistent_drift_count
Best transformation = log1p
Testing angular_coverage
Best transformation = log1p
Testing distance_ratio
Best transformation = log1p
Testing trajectory_linearity
Best transformation = square
Testing mean_similarity
Best transformation = yeo-johnson
Testing std_similarity
Best transformation = original
Testing min_similarity
Best transformation = square
Testing max_similarity
Best transformation = square
Testing interaction_risk
Best transformation = yeo-johnson
Testing pattern_risk
Best transformation = original
Testing progressive_risk
Best transformation = yeo-johnson
Testing prev_progressive
Best transformation = yeo-

In [7]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    ["feature", "score"],
    ascending=[True, False]
).head(20)

,feature,transform,score
36,angular_coverage,log1p,0.152684
39,angular_coverage,yeo-johnson,0.152684
35,angular_coverage,original,0.152573
37,angular_coverage,square,0.152482
38,angular_coverage,binarize,0.087553
41,distance_ratio,log1p,0.094673
40,distance_ratio,original,0.094405
42,distance_ratio,square,0.094245
44,distance_ratio,yeo-johnson,0.094245
43,distance_ratio,binarize,0.035542


In [8]:
best_transforms = (
    results_df
    .sort_values("score", ascending=False)
    .groupby("feature")
    .first()
    .reset_index()
)

best_transforms

,feature,transform,score
0,angular_coverage,log1p,0.152684
1,distance_ratio,log1p,0.094673
2,drift_acceleration,yeo-johnson,0.088463
3,drift_momentum,yeo-johnson,0.102287
4,early_high_risk,original,0.000000
5,interaction_risk,yeo-johnson,0.010537
6,interaction_risk_ema3,original,0.015324
7,late_risk_increase,original,0.000000
8,max_similarity,square,0.073756
9,max_threat_so_far,square,0.004724


In [9]:
final_df = df.copy()

for _, row in best_transforms.iterrows():

    feature = row["feature"]
    transform = row["transform"]

    final_df[feature] = transform_feature(
        final_df[feature],
        transform
    )

final_df.head()

,conv_id,turn_id,label,toxicity_score,threat_score,topic_drift_score,drift_acceleration,origin_drift,drift_momentum,persistent_drift_count,...,threat_diff,toxicity_accel,threat_accel,risk_slope_3,max_toxicity_so_far,max_threat_so_far,mean_risk_so_far,early_high_risk,late_risk_increase,risk_growth_ratio
0,0,0,0,0.001422,0.007502,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000002,0.000062,0.000411,0.020276,0.00286,-0.858921
1,0,1,0,0.000737,0.001295,0.3763,0.000000,0.363924,0.000000,0.693147,...,4.326432e-05,-0.000702,4.326432e-05,2.378080e-04,0.000002,0.000062,0.000158,0.020276,0.00286,-0.858921
2,0,2,0,0.000129,0.000018,0.3800,0.003622,0.575228,0.264810,1.098612,...,1.656827e-06,0.000088,2.798816e-05,9.855687e-05,0.000002,0.000062,0.000073,0.020276,0.00286,-0.858921
3,0,3,0,0.000417,0.000070,0.3180,0.044511,0.671133,0.254883,1.386294,...,2.673073e-09,0.000904,1.792599e-06,3.964113e-05,0.000002,0.000062,0.000045,0.020276,0.00286,-0.858921
4,0,4,0,0.001460,0.000051,0.4204,0.060843,0.645672,0.262211,1.609438,...,3.742132e-10,0.000775,5.047586e-09,1.290806e-08,0.000002,0.000062,0.000040,0.020276,0.00286,-0.858921


In [27]:
best_transforms

,feature,transform,score
0,angular_coverage,log1p,0.152684
1,distance_ratio,log1p,0.094673
2,drift_acceleration,yeo-johnson,0.088463
3,drift_momentum,yeo-johnson,0.102287
4,early_high_risk,original,0.000000
5,interaction_risk,yeo-johnson,0.010537
6,interaction_risk_ema3,original,0.015324
7,late_risk_increase,original,0.000000
8,max_similarity,square,0.073756
9,max_threat_so_far,square,0.004724


In [28]:
import json

result = {
    row["feature"]: row["transform"]
    for _, row in best_transforms.iterrows()
}

json_file_path = "../../integrated/config/feature_transforms.json"

with open(json_file_path, "w") as json_file:
    json.dump(result, json_file, indent=4)

print("Saved:", json_file_path)

Saved: ../../integrated/config/feature_transforms.json
